In [1]:
# build_baseline_features
import pandas as pd
import numpy as np

df = pd.read_csv("event_windows.csv", parse_dates=["date", "event_date"])

rows = []

for event_date, group in df.groupby("event_date"):
    group = group.sort_values("date")

    close = group["close"].values
    volume = group["volume"].values

    returns = np.diff(close) / close[:-1]

    row = {
        "event_date": event_date,
        "pre_return": returns[0],
        "post_1d_return": returns[1],
        "post_3d_return": (close[-1] - close[1]) / close[1],
        "return_std": returns.std(),
        "volume_change": (volume[-1] - volume[1]) / volume[1],
    }

    # 标签
    row["label"] = int(row["post_3d_return"] > 0)
    rows.append(row)

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv("baseline_dataset.csv", index=False)

print("Baseline 数据集完成")
print(baseline_df.head())


Baseline 数据集完成
  event_date  pre_return  post_1d_return  post_3d_return  return_std  \
0 2018-01-30    0.012011        0.046280       -0.003462    0.030151   
1 2018-03-06   -0.003993       -0.017812        0.021539    0.017151   
2 2018-04-03   -0.005079        0.038230        0.071496    0.015912   
3 2018-05-07    0.086924        0.024150        0.036294    0.036479   
4 2018-06-04    0.075480        0.011577       -0.001904    0.033816   

   volume_change  label  
0       0.452891      0  
1      -0.532740      1  
2      -0.203624      1  
3      -0.540910      1  
4      -0.469311      0  


In [5]:
# train_baseline_models
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("baseline_dataset.csv")

X = df.drop(columns=["event_date", "label"])
y = df["label"]

# 标准化（LR / SVM）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

results = {}

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
results["Logistic Regression"] = accuracy_score(y_test, lr.predict(X_test))

# SVM
svm = SVC(kernel="rbf")
svm.fit(X_train, y_train)
results["SVM"] = accuracy_score(y_test, svm.predict(X_test))

# Random Forest（不需要标准化）
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results["Random Forest"] = accuracy_score(y_test, rf.predict(X_test))

print("Baseline 模型对比结果")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Baseline 模型对比结果
Logistic Regression: 0.9000
SVM: 0.8000
Random Forest: 1.0000


In [4]:
# compare_models
import pandas as pd

results = {
    "Model": [
        "Logistic Regression",
        "SVM",
        "Random Forest",
        "LSTM (Event-driven)"
    ],
    "Accuracy": [
        1.00,
        0.90,
        0.80,
        0.75,
    ]
}

df = pd.DataFrame(results)
df.to_csv("model_comparison.csv", index=False)
print(df)

                 Model  Accuracy
0  Logistic Regression      1.00
1                  SVM      0.90
2        Random Forest      0.80
3  LSTM (Event-driven)      0.75
